# EdgeAI Hybrid Approach – Desktop Vision with Live Webcam

This notebook extends the desktop vision prototyping workflow by **connecting to a live desktop camera**.

## Purpose
- Use a USB / laptop webcam as a stand-in for an embedded camera
- Practice real-time image acquisition and processing
- Introduce **frame-rate limits** similar to OpenMV constraints

This keeps the mental model aligned with embedded vision systems.

## Requirements

### Software
- Python 3.9+
- Jupyter Notebook or JupyterLab
- OpenCV, NumPy, Matplotlib

Install once:
```bash
pip install opencv-python numpy matplotlib jupyter
```

### Notes
- macOS / Windows: camera usually works out of the box
- Linux: camera access via `/dev/video*` (V4L2)

## Accessing the Desktop Camera

OpenCV uses `VideoCapture` as a cross-platform abstraction over the system camera.

Conceptually, this is equivalent to acquiring frames from an embedded sensor.

In [ ]:
# Open the default camera (0)
import cv2

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError('Camera not accessible')

print('Camera opened successfully')

# Read one frame
ret, frame = cap.read()
cap.release()

print('Frame captured:', frame.shape if ret else 'Failed')

: 

## Displaying a Single Frame (Notebook-Safe)

`cv2.imshow()` is unreliable inside Jupyter. Instead, display frames inline.

In [ ]:
import matplotlib.pyplot as plt

frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

plt.imshow(frame_rgb)
plt.axis('off')
plt.title('Single Camera Frame')

## Live Camera Loop (Limited Frames)

This simulates frame-by-frame processing similar to an embedded vision loop.
The loop is intentionally limited to avoid overwhelming the notebook.

In [ ]:
from IPython.display import clear_output

cap = cv2.VideoCapture(0)

for i in range(30):  # capture 30 frames
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    clear_output(wait=True)
    plt.imshow(frame_rgb)
    plt.axis('off')
    plt.title(f'Live Frame {i}')

cap.release()

## Adding Vision Processing (Edges)

This represents lightweight per-frame processing suitable for embedded targets.

In [ ]:
cap = cv2.VideoCapture(0)

for i in range(30):
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)

    clear_output(wait=True)
    plt.imshow(edges, cmap='gray')
    plt.axis('off')
    plt.title('Edge Detection')

cap.release()

## Frame-Rate Limiting (OpenMV-Style)

Embedded systems often run at **5–30 FPS**.
Here we explicitly control the frame rate using time delays.

In [ ]:
import time

TARGET_FPS = 5
FRAME_TIME = 1.0 / TARGET_FPS

cap = cv2.VideoCapture(0)

for i in range(20):
    start = time.time()

    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    clear_output(wait=True)
    plt.imshow(gray, cmap='gray')
    plt.axis('off')
    plt.title(f'Frame {i} @ {TARGET_FPS} FPS')

    elapsed = time.time() - start
    sleep_time = max(0, FRAME_TIME - elapsed)
    time.sleep(sleep_time)

cap.release()

## Key Takeaways

- Desktop webcams can fully replace embedded cameras *for learning*
- Frame-by-frame processing matches embedded execution
- Artificial FPS limits teach real-time constraints
- Code structure transfers directly to microcontroller vision pipelines

Once hardware is available, only the camera API changes.